# ATML PA2 — Task 5: RLVR versus RLAIF
Use an A100 runtime. No training is required. This evaluates untouched SFT and the two course-supplied math adapters, not your Task 3 adapter. Task 4's manual audit can remain pending. The original math prompts already request reasoning and a designated final answer; they are used unchanged.

Full run: 300 GSM8K + 100 SVAMP problems for each of three policies (1,200 responses), 800 policy-vs-SFT judge pairs and 100 diagnostic pairs. Outputs persist in MyDrive/PA2. Runtime depends on response length; allow several hours and use the live progress estimates.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, shutil, json, signal, zipfile

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/PA2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['PA2_ARTIFACT_ROOT'] = str(DRIVE_ROOT)
os.environ['USE_TF'] = '0'
os.environ['PYTHONUNBUFFERED'] = '1'
REPO = Path('/content/ATML')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WasifAhmed1917/ATML.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
PA2 = REPO / 'PA2'
# If Task 5 is not yet published, upload ATML_PA2_Task5.zip once. Keep
# the package in Drive so reconnecting restores exactly the same source.
PACKAGE = DRIVE_ROOT / 'ATML_PA2_Task5.zip'
if not (PA2 / 'task5_feedback/run_all.py').exists():
    if not PACKAGE.exists():
        print('Upload ATML_PA2_Task5.zip supplied with this notebook.')
        uploaded = files.upload()
        assert len(uploaded) == 1
        name, content = next(iter(uploaded.items()))
        assert name.endswith('.zip')
        PACKAGE.write_bytes(content)
    with zipfile.ZipFile(PACKAGE) as archive:
        for member in archive.infolist():
            relative = Path(member.filename)
            if not relative.parts or relative.parts[0] != 'PA2' or member.is_dir():
                continue
            destination = (REPO / relative).resolve()
            assert destination.is_relative_to(PA2.resolve()), 'Unsafe archive path'
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(member))
os.chdir(PA2)
assert (PA2 / 'task5_feedback/run_all.py').exists()
print('Repository revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('Persistent artifacts:', DRIVE_ROOT)


## Dependencies and live logs
The checked-in requirements.txt is unchanged from the release. After a disconnect, rerun setup, then the unfinished stage with --resume. Keep the same code, software and GPU while resuming.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

def run(module, *args, artifact_root=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'USE_TF': '0'}
    if artifact_root is not None: env['PA2_ARTIFACT_ROOT'] = str(artifact_root)
    log_dir = DRIVE_ROOT / 'logs/task5_feedback'
    log_dir.mkdir(parents=True, exist_ok=True)
    label = module.replace('.', '_') + ('_' + args[args.index('--stage')+1] if '--stage' in args else '')
    command = [sys.executable, '-u', '-m', module, *args]
    print('Running:', ' '.join(command), flush=True)
    with (log_dir / (label + '.log')).open('a') as log:
        process = subprocess.Popen(command, cwd=PA2, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            try: os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError: pass
            process.wait()
            raise
    print('Exit code:', status)
    if status: raise subprocess.CalledProcessError(status, command)
run('scripts.check_environment')


## Fixed assets and checks
Downloads only Task 5's data and supplied adapters from the same pinned course asset revision. Uses the fixed 300-example GSM8K test subset, unchanged 100-example SVAMP file and 100 staff-validated diagnostic responses. No random resampling.

In [ ]:
run('task5_feedback.download_assets')
run('task5_feedback.validate_data')
run('unittest', 'discover', '-s', 'tests', '-p', 'test_task5*.py', '-v')


## Optional isolated smoke test
Two GSM8K prompts per policy, with the full released 512-token cap and sampling settings. Test outputs live under a separate directory and cannot satisfy full-run completion checks.

In [ ]:
SMOKE_ROOT = DRIVE_ROOT / 'task5_smoke'
for policy in ('sft', 'rlvr', 'rlaif'):
    run('task5_feedback.generate_responses', '--dataset', 'gsm', '--policy', policy,
        '--smoke', '--resume', artifact_root=SMOKE_ROOT)
for policy in ('rlvr', 'rlaif'):
    run('task5_feedback.judge_pairs', '--dataset', 'gsm', '--policy', policy,
        '--smoke', '--resume', artifact_root=SMOKE_ROOT)


## In-domain: GSM8K
Same fixed messages, seed, batch size 4, temperature 0.7, top_p 0.9, sampling enabled and 512 new tokens for every policy, as configured in the release. Prompts are retained in full. Partially saved batches replay with their original seed and membership after a disconnect. Judging uses the supplied deterministic, hash-balanced pairwise judge with its original four-token cap.

In [ ]:
run('task5_feedback.run_all', '--stage', 'gsm', '--resume')


## Controlled reward diagnostics
Scores clean versus each supplied variant for all 20 problems, with the clean category represented by an identity self-pair. Reports exact-verifier and AI-judge better/tie/wrong rates, reasoning sensitivity, outcome sensitivity, filler preference and distractor robustness. These are already staff-validated responses; no human labeling step is required.

In [ ]:
run('task5_feedback.run_all', '--stage', 'diagnostics', '--resume')


## Transfer: SVAMP
Same three policies and evaluation protocol. No retraining or tuning.

In [ ]:
run('task5_feedback.run_all', '--stage', 'transfer', '--resume')


## Tables, agreement, failure cases, costs and report figures
Exports only complete full-set results. Exact correctness and strict final-field formatting are separate metrics. Verifier ties on reasoning-only changes are legitimate invariance, not errors. A negative accuracy drop means SVAMP accuracy is higher; benchmark difficulty differs.

In [ ]:
run('task5_feedback.run_all', '--stage', 'export', '--resume')


In [ ]:
import pandas as pd
from IPython.display import display, Image
RESULTS = DRIVE_ROOT / 'results/task5_feedback'
for name in ('summary.csv', 'transfer_drops.csv', 'diagnostic_rates.csv', 'sensitivity.csv', 'failure_types.csv'):
    display(pd.read_csv(RESULTS / name))
print(json.loads((RESULTS / 'completion_checks.json').read_text()))
for path in sorted((RESULTS / 'plots').glob('*.png')): display(Image(filename=str(path)))
display(pd.read_csv(RESULTS / 'qualitative_diagnostics.csv'))
print((RESULTS / 'evidence_notes.md').read_text())


## Save and download
Use qualitative_diagnostics.csv to explain reasoning corruption, filler and wrong-final/gold-distractor disagreements. Review these examples before making claims about reasoning or judge errors. The ZIP contains generated records, fixed input IDs, metrics, qualitative pairs, cost metadata and plots. Include logs and machine-readable evidence in your final submission; do not use these held-out results to tune earlier training.

In [ ]:
archive = shutil.make_archive('/content/ATML_PA2_T5_results', 'zip', root_dir=DRIVE_ROOT / 'results', base_dir='task5_feedback')
shutil.copy2(archive, DRIVE_ROOT / Path(archive).name)
files.download(archive)
